# Ground Truth Sanity Check

Loads the three benchmark datasets (WISDM, MIT ECG, Bus), runs the deterministic ground-truth builder for each, and visualizes the results for sanity checking. Also diffs the freshly-computed answers against the saved JSON artifacts.

In [ ]:
import sys
import json
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Make sure the flashfusion package is importable when running from eval/
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from flashfusion.pipeline.loader import load_dataset_by_name
from flashfusion.eval.ground_truth_builder import (
    build_ground_truth_wisdm,
    build_ground_truth_mit_ecg,
    build_ground_truth_bus,
 )
from flashfusion.eval.queries import DATASET_WISDM, DATASET_MIT_ECG, DATASET_BUS

# ── Paths ─────────────────────────────────────────────────────────────────────
WISDM_DATA   = REPO_ROOT / "data/AutoIOT_dataset/IMU/WISDM_ar_v1.1_raw.txt"
ECG_DATA     = REPO_ROOT / "data/AutoIOT_dataset/ECG.0/MIT_arrythmia_v1.txt"
BUS_DATA     = REPO_ROOT / "data/bus/bus_data.csv"

WISDM_GT_JSON = REPO_ROOT / "flashfusion/eval/ground_truth/ground_truth_wisdm.json"
ECG_GT_JSON   = REPO_ROOT / "flashfusion/eval/ground_truth/ground_truth_mit_ecg.json"
BUS_GT_JSON   = REPO_ROOT / "flashfusion/eval/ground_truth/ground_truth_bus.json"

# ── ECG fast-load config ──────────────────────────────────────────────────────
ECG_TARGET_RECORDS = {101, 106, 208, 221, 234}
ECG_CHUNKSIZE = 1_000_000
ECG_MAX_ROWS_PER_RECORD = None  # set to an int for extra speed


def load_mit_arrythmia_fast(
    path: Path,
    target_records: set[int],
    chunksize: int = 1_000_000,
    max_rows_per_record: int | None = None,
 ) -> pd.DataFrame:
    print(f"Loading and filtering ECG data in chunks of {chunksize:,}...")
    chunks: list[pd.DataFrame] = []
    counts = {rid: 0 for rid in target_records}

    reader = pd.read_csv(
        path,
        header=None,
        sep="[,;]",
        engine="python",
        on_bad_lines="skip",
        chunksize=chunksize,
    )

    for i, chunk in enumerate(reader):
        chunk = chunk.iloc[:, :6]
        chunk.columns = ["sample_idx", "time_s", "MLII", "V1", "record_id", "annotation"]
        chunk["record_id"] = pd.to_numeric(chunk["record_id"], errors="coerce")

        filtered_chunk = chunk[chunk["record_id"].isin(target_records)].copy()
        if filtered_chunk.empty:
            continue

        filtered_chunk["sample_idx"] = pd.to_numeric(filtered_chunk["sample_idx"], errors="coerce")
        filtered_chunk["record_id"] = filtered_chunk["record_id"].astype("int32")

        if max_rows_per_record is not None:
            keep_parts = []
            for rid, sub in filtered_chunk.groupby("record_id"):
                remaining = max_rows_per_record - counts[int(rid)]
                if remaining <= 0:
                    continue
                take = sub.head(remaining)
                counts[int(rid)] += len(take)
                keep_parts.append(take)
            filtered_chunk = pd.concat(keep_parts, ignore_index=True) if keep_parts else filtered_chunk.iloc[:0]
        else:
            for rid, sub in filtered_chunk.groupby("record_id"):
                counts[int(rid)] += len(sub)

        if not filtered_chunk.empty:
            chunks.append(filtered_chunk)

        if (i + 1) % 5 == 0:
            kept = sum(counts.values())
            print(f"  Processed chunk {i + 1}: kept {kept:,} rows so far")

    final_df = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(
        columns=["sample_idx", "time_s", "MLII", "V1", "record_id", "annotation"]
    )
    print(f"Total rows loaded after filtering: {len(final_df):,}")
    return final_df


print("Repo root:", REPO_ROOT)
for p in [WISDM_DATA, ECG_DATA, BUS_DATA]:
    status = "OK" if p.exists() else "MISSING"
    print(f"  {status:<7} {p.relative_to(REPO_ROOT)}")

Repo root: /Users/kausar/Documents/ff-context/flash-fusion
  OK      data/AutoIOT_dataset/IMU/WISDM_ar_v1.1_raw.txt
  OK      data/AutoIOT_dataset/ECG.0/MIT_arrythmia_v1.txt
  OK      data/bus/bus_data.csv


## 1. Load Datasets

In [3]:
# ── WISDM ─────────────────────────────────────────────────────────────────────
df_wisdm = load_dataset_by_name(str(WISDM_DATA), DATASET_WISDM)
print(f"WISDM  — rows: {len(df_wisdm):,}  columns: {list(df_wisdm.columns)}")
df_wisdm.head(3)

WISDM  — rows: 1,098,198  columns: ['subject_id', 'activity_label', 'timestamp', 'x', 'y', 'z']


,subject_id,activity_label,timestamp,x,y,z
0,33,Jogging,49105962326000,-0.694638,12.680544,0.503953
1,33,Jogging,49106062271000,5.012288,11.264028,0.953424
2,33,Jogging,49106112167000,4.903325,10.882658,-0.081722


In [4]:
# ── MIT ECG (chunk-filtered) ──────────────────────────────────────────────────
df_ecg = load_mit_arrythmia_fast(
    ECG_DATA,
    target_records=ECG_TARGET_RECORDS,
    chunksize=ECG_CHUNKSIZE,
    max_rows_per_record=ECG_MAX_ROWS_PER_RECORD,
)

present_ids = sorted(df_ecg["record_id"].dropna().unique().tolist())
missing_ids = sorted(ECG_TARGET_RECORDS - set(present_ids))
print(f"MIT ECG — rows: {len(df_ecg):,}  columns: {list(df_ecg.columns)}")
print(f"          record_ids present: {present_ids}")
if missing_ids:
    print(f"          missing record_ids: {missing_ids}")
if ECG_MAX_ROWS_PER_RECORD is not None:
    print(f"          NOTE: sampling enabled (max_rows_per_record={ECG_MAX_ROWS_PER_RECORD})")
else:
    print("          NOTE: filtered to target record_ids only")

df_ecg.head(3)

Loading and filtering ECG data in chunks of 1,000,000...
  Processed chunk 15: kept 1,950,000 rows so far
Total rows loaded after filtering: 3,250,000
MIT ECG — rows: 3,250,000  columns: ['sample_idx', 'time_s', 'MLII', 'V1', 'record_id', 'annotation']
          record_ids present: [101, 106, 208, 221, 234]
          NOTE: filtered to target record_ids only


,sample_idx,time_s,MLII,V1,record_id,annotation
0,0,0.000000,-0.345,-0.16,101,NaN
1,1,0.002778,-0.345,-0.16,101,NaN
2,2,0.005556,-0.345,-0.16,101,NaN


In [5]:
# ── Bus ───────────────────────────────────────────────────────────────────────
df_bus = load_dataset_by_name(str(BUS_DATA), DATASET_BUS)
print(f"Bus    — rows: {len(df_bus):,}  columns: {list(df_bus.columns)}")
df_bus.head(3)

Bus    — rows: 1,219  columns: ['timestamp', 'latitude', 'longitude', 'accel_mean', 'accel_variance', 'accel_stats_x_p1', 'accel_stats_x_p10', 'accel_stats_x_p90', 'accel_stats_x_p99', 'accel_stats_y_p1', 'accel_stats_y_p10', 'accel_stats_y_p90', 'accel_stats_y_p99', 'accel_stats_z_p1', 'accel_stats_z_p10', 'accel_stats_z_p90', 'accel_stats_z_p99']


,timestamp,latitude,longitude,accel_mean,accel_variance,accel_stats_x_p1,accel_stats_x_p10,accel_stats_x_p90,accel_stats_x_p99,accel_stats_y_p1,accel_stats_y_p10,accel_stats_y_p90,accel_stats_y_p99,accel_stats_z_p1,accel_stats_z_p10,accel_stats_z_p90,accel_stats_z_p99
0,2025-06-06 16:36:34,33.776970,-84.389880,9.344,0.127,-1.686,-0.46,1.073,1.992,0.766,2.452,3.065,3.218,8.274,8.581,9.194,11.032
1,2025-06-06 16:36:31,33.776971,-84.389857,9.344,0.127,-1.686,-0.46,1.073,1.992,0.766,2.452,3.065,3.218,8.274,8.581,9.194,11.032
2,2025-06-06 16:36:28,33.776974,-84.389839,9.344,0.127,-1.686,-0.46,1.073,1.992,0.766,2.452,3.065,3.218,8.274,8.581,9.194,11.032


### Ground truth printing

In [ ]:
# ── WISDM ground truth (inline) ───────────────────────────────────────────────
_df = df_wisdm.copy()
_df["activity_label"] = _df["activity_label"].astype(str).str.strip()
_df["activity_lower"] = _df["activity_label"].str.lower()
_df["magnitude"] = (_df["x"] ** 2 + _df["y"] ** 2 + _df["z"] ** 2) ** 0.5

# ── Q1  [direct / FILTER+AGGREGATE]
# "What is the maximum recorded x-acceleration for user 15?"
q1 = float(_df.loc[_df["subject_id"] == 15, "x"].max())
print(f"Q1  [ANSWER]  Maximum x-acceleration for user 15 is {q1:.4f}.")

# ── Q2  [direct / FILTER+COUNT]
# "How many total samples in the dataset are classified as the Walking activity?"
q2 = int(_df.loc[_df["activity_lower"] == "walking"].shape[0])
print(f"Q2  [ANSWER]  Total Walking samples in the dataset: {q2}.")

# ── Q3  [direct / FILTER+AGGREGATE]
# "What is the average y-accel value for user 5 during the Sitting activity?"
q3 = float(_df.loc[(_df["subject_id"] == 5) & (_df["activity_lower"] == "sitting"), "y"].mean())
print(f"Q3  [ANSWER]  Average y-acceleration for user 5 during Sitting is {q3:.4f}.")

# ── Q4  [direct / GROUPBY+RANK]
# "Which user has the highest total number of recorded data samples?"
_per_user = _df.groupby("subject_id").size().sort_values(ascending=False)
q4_user, q4_count = int(_per_user.index[0]), int(_per_user.iloc[0])
print(f"Q4  [ANSWER]  User with the highest sample count is {q4_user} with {q4_count} samples.")

# ── Q5  [intermediate / FILTER+AGGREGATE]
# "Compare the overall acceleration magnitude between dynamic movements … and resting states …"
_dyn_mask  = _df["activity_lower"].isin({"walking", "jogging", "upstairs", "downstairs"})
_rest_mask = _df["activity_lower"].isin({"sitting", "standing"})
q5_dyn  = float(_df.loc[_dyn_mask,  "magnitude"].mean())
q5_rest = float(_df.loc[_rest_mask, "magnitude"].mean())
q5_diff = q5_dyn - q5_rest
print(f"Q5  [ANSWER]  Avg magnitude dynamic={q5_dyn:.4f}, resting={q5_rest:.4f}, diff={q5_diff:.4f}.")

# ── Q6  [intermediate / FILTER+GROUPBY+COMPARE]
# "Identify the user whose total recorded duration of stationary activities exceeds locomotion."
_ds = _df.sort_values(["subject_id", "timestamp"]).copy()
_ds["dt_s"] = (_ds.groupby("subject_id")["timestamp"].diff().clip(lower=0).fillna(0) / 1e9)
_stat_s = _ds.loc[_ds["activity_lower"].isin({"sitting", "standing"})].groupby("subject_id")["dt_s"].sum()
_loco_s = _ds.loc[_ds["activity_lower"].isin({"walking", "jogging", "upstairs", "downstairs"})].groupby("subject_id")["dt_s"].sum()
_q6 = (pd.DataFrame({"stat": _stat_s, "loco": _loco_s}).fillna(0)
         .assign(delta=lambda r: r["stat"] - r["loco"])
         .query("delta > 0").sort_values("delta", ascending=False))
q6_users = [int(u) for u in _q6.index.tolist()]
if q6_users:
    print(f"Q6  [ANSWER]  Users {q6_users} have stationary > locomotion. "
          f"Largest margin: user {q6_users[0]} (delta={_q6.iloc[0]['delta']:.2f}s).")
else:
    print("Q6  [ANSWER]  No user has stationary > locomotion.")

# ── Q7  [intermediate / FILTER+DERIVE+AGGREGATE]
# "What is the median net acceleration vector length for user 20 while ascending steps?"
q7 = float(_df.loc[(_df["subject_id"] == 20) & (_df["activity_label"] == "Upstairs"), "magnitude"].median())
print(f"Q7  [ANSWER]  Median net accel vector length for user 20 ascending steps is {q7:.4f}.")

# ── Q8  [intermediate / FILTER+AGGREGATE+DIFF]
# "Calculate the difference in average z-axis acceleration between ascending and descending …"
q8_up   = float(_df.loc[_df["activity_lower"].isin({"upstairs"}), "z"].mean())
q8_down = float(_df.loc[_df["activity_lower"] == "downstairs", "z"].mean())
print(f"Q8  [ANSWER]  Avg z ascending={q8_up:.4f}, descending={q8_down:.4f}, diff={q8_up - q8_down:.4f}.")

# ── Q9–Q12  [out_of_scope — expected rejections]
print("Q9  [REJECT]  Speed in mph and user age are not available in this dataset.")
print("Q10 [REJECT]  Acceleration records do not contain geographic location signals.")
print("Q11 [REJECT]  Sex and cadence attributes are unavailable in this dataset.")
print("Q12 [REJECT]  Personalized workout recommendation is outside benchmark analytics scope.")

Q1  [ANSWER]  Maximum x-acceleration for user 15 is 19.5700.
Q2  [ANSWER]  Total Walking samples in the dataset: 424397.
Q3  [ANSWER]  Average y-acceleration for user 5 during Sitting is 3.0414.
Q4  [ANSWER]  User with the highest sample count is 20 with 56632 samples.
Q5  [ANSWER]  Avg magnitude dynamic=11.9625, resting=9.8328, diff=2.1297.
Q6  [ANSWER]  Users [20, 6, 16] have stationary > locomotion. Largest margin: user 20 (delta=54004.34s).
Q7  [ANSWER]  Median net accel vector length for user 20 ascending steps is 9.8931.
Q8  [ANSWER]  Avg z ascending=0.3235, descending=0.6841, diff=-0.3605.
Q9  [REJECT]  Speed in mph and user age are not available in this dataset.
Q10 [REJECT]  Acceleration records do not contain geographic location signals.
Q11 [REJECT]  Sex and cadence attributes are unavailable in this dataset.
Q12 [REJECT]  Personalized workout recommendation is outside benchmark analytics scope.


In [10]:
df_wisdm["activity_label"].unique()

array(['Jogging', 'Walking', 'Upstairs', 'Downstairs', 'Sitting',
       'Standing'], dtype=object)

In [7]:
# ── MIT ECG ground truth (inline) ─────────────────────────────────────────────
_ecg = df_ecg.copy()
_ecg["annotation"] = _ecg["annotation"].astype(str).fillna("")
_ecg["is_annotated"] = _ecg["annotation"].str.strip() != ""

# ── Q1  [direct / FILTER+AGGREGATE]
# "What is the minimum MLII value recorded for record_id 101?"
q1 = float(_ecg.loc[_ecg["record_id"] == 101, "MLII"].min())
print(f"Q1  [ANSWER]  Minimum MLII for record_id 101 is {q1:.4f}.")

# ── Q2  [direct / FILTER+AGGREGATE]
# "What is the total recording duration in seconds (maximum time_s) for record_id 234?"
q2 = float(_ecg.loc[_ecg["record_id"] == 234, "time_s"].max())
print(f"Q2  [ANSWER]  Total recording duration for record_id 234 is {q2:.4f} seconds.")

# ── Q3  [direct / FILTER+COUNT]
# "How many samples in record_id 106 have an MLII value greater than 0?"
q3 = int(_ecg.loc[(_ecg["record_id"] == 106) & (_ecg["MLII"] > 0)].shape[0])
print(f"Q3  [ANSWER]  record_id 106 has {q3} samples with MLII > 0.")

# ── Q4  [direct / FILTER+AGGREGATE]
# "What is the timestamp (time_s) of the very last annotated beat in record_id 221?"
q4 = float(_ecg.loc[(_ecg["record_id"] == 221) & (_ecg["is_annotated"]), "time_s"].max())
print(f"Q4  [ANSWER]  Last annotated beat in record_id 221 at time_s = {q4:.6f}.")

# ── Q5  [intermediate / FILTER+AGGREGATE+DERIVE]
# "Estimate the average heart rate in BPM for record_id 208 based on annotation count and max time_s."
_r208 = _ecg.loc[_ecg["record_id"] == 208]
q5_ann  = int(_r208.loc[_r208["is_annotated"]].shape[0])
q5_dur  = float(_r208["time_s"].max())
q5_bpm  = (q5_ann / q5_dur) * 60.0
print(f"Q5  [ANSWER]  record_id 208: {q5_ann} beats over {q5_dur:.4f}s → est. HR = {q5_bpm:.2f} BPM.")

# ── Q6  [intermediate / GROUPBY+AGGREGATE+DERIVE+RANK]
# "Which record_id exhibits the largest peak-to-peak MLII amplitude?"
_mlii_range = (_ecg.groupby("record_id")["MLII"]
               .agg(lambda x: x.max() - x.min())
               .sort_values(ascending=False))
q6_rec   = int(_mlii_range.index[0])
q6_range = float(_mlii_range.iloc[0])
print(f"Q6  [ANSWER]  record_id {q6_rec} has the largest peak-to-peak MLII amplitude at {q6_range:.4f}.")

# ── Q7  [intermediate / DERIVE+FILTER+GROUPBY+RANK]
# "For record_id 101, which 10-second interval contains the highest number of annotated beats?"
_r101_ann = _ecg.loc[(_ecg["record_id"] == 101) & (_ecg["is_annotated"])].copy()
_r101_ann["interval_10s"] = (_r101_ann["time_s"] // 10).astype(int)
_interval_counts = _r101_ann.groupby("interval_10s").size().sort_values(ascending=False)
q7_bin   = int(_interval_counts.index[0])
q7_start = q7_bin * 10
q7_count = int(_interval_counts.iloc[0])
print(f"Q7  [ANSWER]  Interval [{q7_start}s, {q7_start + 10}s) has most annotated beats for record_id 101: {q7_count} beats.")

# ── Q8  [intermediate / FILTER+DERIVE+AGGREGATE]
# "Calculate the root mean square (RMS) of the MLII signal for record_id 106."
_mlii_106 = _ecg.loc[_ecg["record_id"] == 106, "MLII"]
q8_rms = float(np.sqrt((_mlii_106 ** 2).mean()))
print(f"Q8  [ANSWER]  RMS of MLII signal for record_id 106 is {q8_rms:.4f}.")

# ── Q9–Q12  [out_of_scope — expected rejections]
print("Q9  [REJECT]  Patient outcome and mortality data are unavailable in this ECG dataset.")
print("Q10 [REJECT]  BMI and anthropometric metadata are unavailable in this ECG dataset.")
print("Q11 [REJECT]  Family medical history is unavailable in this ECG dataset.")
print("Q12 [REJECT]  Hemodynamic variables such as blood pressure are unavailable in this ECG dataset.")

Q1  [ANSWER]  Minimum MLII for record_id 101 is -3.1750.
Q2  [ANSWER]  Total recording duration for record_id 234 is 1805.5528 seconds.
Q3  [ANSWER]  record_id 106 has 111769 samples with MLII > 0.
Q4  [ANSWER]  Last annotated beat in record_id 221 at time_s = 1805.552778.
Q5  [ANSWER]  record_id 208: 650000 beats over 1805.5528s → est. HR = 21600.03 BPM.
Q6  [ANSWER]  record_id 208 has the largest peak-to-peak MLII amplitude at 7.1350.
Q7  [ANSWER]  Interval [0s, 10s) has most annotated beats for record_id 101: 3600 beats.
Q8  [ANSWER]  RMS of MLII signal for record_id 106 is 0.4088.
Q9  [REJECT]  Patient outcome and mortality data are unavailable in this ECG dataset.
Q10 [REJECT]  BMI and anthropometric metadata are unavailable in this ECG dataset.
Q11 [REJECT]  Family medical history is unavailable in this ECG dataset.
Q12 [REJECT]  Hemodynamic variables such as blood pressure are unavailable in this ECG dataset.


In [8]:
# ── Bus ground truth (inline) ─────────────────────────────────────────────────
_bus = df_bus.copy()
_bus["timestamp"] = pd.to_datetime(_bus["timestamp"], errors="coerce")
_bus = _bus.dropna().reset_index(drop=True)

# ── Q1  [direct / AGGREGATE]
# "What is the maximum accel_variance observed in this dataset?"
q1 = float(_bus["accel_variance"].max())
print(f"Q1  [ANSWER]  Maximum accel_variance is {q1:.4f}.")

# ── Q2  [direct / AGGREGATE]
# "What is the average accel_mean across all recorded samples?"
q2 = float(_bus["accel_mean"].mean())
print(f"Q2  [ANSWER]  Average accel_mean across all samples is {q2:.4f}.")

# ── Q3  [direct / RANK+SELECT]
# "At what exact timestamp was the highest accel_stats_z_p99 recorded?"
q3_idx = int(_bus["accel_stats_z_p99"].idxmax())
q3_ts  = _bus.loc[q3_idx, "timestamp"]
q3_val = float(_bus.loc[q3_idx, "accel_stats_z_p99"])
print(f"Q3  [ANSWER]  Highest accel_stats_z_p99 ({q3_val:.4f}) at {q3_ts.strftime('%Y-%m-%d %H:%M:%S')}.")

# ── Q4  [direct / FILTER+COUNT]
# "How many data samples show an accel_variance strictly greater than 0.20?"
q4 = int((_bus["accel_variance"] > 0.20).sum())
print(f"Q4  [ANSWER]  Samples with accel_variance > 0.20: {q4}.")

# ── Q5  [intermediate / DERIVE+FILTER+AGGREGATE+COMPARE]
# "Is the northern half of the route (latitude above median) rougher … based on average acceleration variance?"
_lat_med = float(_bus["latitude"].median())
_north   = _bus["latitude"] >= _lat_med
q5_north = float(_bus.loc[_north,  "accel_variance"].mean())
q5_south = float(_bus.loc[~_north, "accel_variance"].mean())
q5_diff  = q5_north - q5_south
print(f"Q5  [ANSWER]  Median lat={_lat_med:.6f}. North={q5_north:.4f}, South={q5_south:.4f} → "
      f"northern half is {'rougher' if q5_diff > 0 else 'smoother'} by {abs(q5_diff):.4f}.")

# ── Q6  [intermediate / DERIVE+RANK+SELECT]
# "Identify the exact coordinates of the largest vertical shock (z_p99 - z_p1)."
_bus["vertical_shock"] = _bus["accel_stats_z_p99"] - _bus["accel_stats_z_p1"]
q6_idx = int(_bus["vertical_shock"].idxmax())
q6_row = _bus.loc[q6_idx]
print(f"Q6  [ANSWER]  Largest vertical shock ({float(q6_row['vertical_shock']):.4f}) at "
      f"({float(q6_row['latitude']):.6f}, {float(q6_row['longitude']):.6f}), "
      f"timestamp {q6_row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')}.")

# ── Q7  [intermediate / DERIVE+AGGREGATE]
# "Calculate the average overall magnitude of peak acceleration using the 99th percentiles of X, Y, Z."
_bus["peak_magnitude"] = np.sqrt(
    _bus["accel_stats_x_p99"] ** 2 +
    _bus["accel_stats_y_p99"] ** 2 +
    _bus["accel_stats_z_p99"] ** 2
)
q7 = float(_bus["peak_magnitude"].mean())
print(f"Q7  [ANSWER]  Average 3D peak magnitude [sqrt(x_p99²+y_p99²+z_p99²)] is {q7:.4f}.")

# ── Q8  [intermediate / DERIVE+GROUPBY+AGGREGATE+RANK]
# "Which 1-minute interval experienced the most sustained turbulence (highest total accel_variance)?"
_bus["minute_bin"] = _bus["timestamp"].dt.floor("min")
_var_by_min = _bus.groupby("minute_bin")["accel_variance"].sum().sort_values(ascending=False)
q8_bin   = _var_by_min.index[0]
q8_total = float(_var_by_min.iloc[0])
print(f"Q8  [ANSWER]  1-minute window {q8_bin.strftime('%Y-%m-%d %H:%M:%S')} had highest total accel_variance={q8_total:.4f}.")

# ── Q9–Q12  [out_of_scope — expected rejections]
print("Q9  [REJECT]  Passenger occupancy data is unavailable in this bus dataset.")
print("Q10 [REJECT]  Weather metadata is unavailable in this bus dataset.")
print("Q11 [REJECT]  Driver identity metadata is unavailable in this bus dataset.")
print("Q12 [REJECT]  Future road maintenance labels are unavailable in this bus dataset.")

Q1  [ANSWER]  Maximum accel_variance is 5.8690.
Q2  [ANSWER]  Average accel_mean across all samples is 9.2834.
Q3  [ANSWER]  Highest accel_stats_z_p99 (16.7020) at 2025-06-06 16:02:01.
Q4  [ANSWER]  Samples with accel_variance > 0.20: 352.
Q5  [ANSWER]  Median lat=33.776676. North=0.3699, South=0.1631 → northern half is rougher by 0.2068.
Q6  [ANSWER]  Largest vertical shock (11.1860) at (33.776932, -84.391906), timestamp 2025-06-06 16:02:01.
Q7  [ANSWER]  Average 3D peak magnitude [sqrt(x_p99²+y_p99²+z_p99²)] is 11.4585.
Q8  [ANSWER]  1-minute window 2025-06-06 16:01:00 had highest total accel_variance=88.0350.
Q9  [REJECT]  Passenger occupancy data is unavailable in this bus dataset.
Q10 [REJECT]  Weather metadata is unavailable in this bus dataset.
Q11 [REJECT]  Driver identity metadata is unavailable in this bus dataset.
Q12 [REJECT]  Future road maintenance labels are unavailable in this bus dataset.
